# Módulo 01: Fundamentos de Great Expectations (Contexto Efímero)

En lugar de fragmentar los conceptos, en este notebook cubriremos las bases principales del flujo de trabajo de Great Expectations (GX). Este notebook incluye el contenido de tres lecciones previas traducidas y unificadas:
- Generación del flujo básico con Pandas y Ephemeral Context.
- Interpretación detallada de los resultados de validación.
- Personalización de formatos de resultado (Result Format).

### Jerarquía de Conceptos de GX
```mermaid
graph TD
    A[Data Context] --> B[Data Source]
    B --> C[Data Asset]
    C --> D[Batch Definition]
    D --> E[Batch / Lote de Datos]
    
    F[Expectation Suite] --> G[Expectation 1]
    F --> H[Expectation 2]
    
    D -. Se une con .-> I[Validation Definition]
    F -. Se une con .-> I
    
    I -- Ejecuta --> J[Validation Result / Data Docs]
```

### 1. Importar Librerías

In [9]:
import great_expectations as gx
import pandas as pd

### 2. Creación del Contexto (Ephemeral)
Un contexto *Ephemeral* vive solo en memoria. Es ideal para exploración rápida, ya que no genera carpetas físicas persistentes.

In [10]:
context = gx.get_context(mode="ephemeral")
print("¡Contexto GX inicializado!")

¡Contexto GX inicializado!


### 3. Configuración de Puntos de Datos (Data Source, Data Asset, Batch Definition)
En esta prueba usaremos los datos de ventas.

In [11]:
# DataSource
data_source_name = 'fuente_datos_ventas'
data_source = context.data_sources.add_pandas(name=data_source_name)

# DataAsset
data_asset_name = 'activo_ventas_ecommerce'
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

# Batch Definition
batch_definition_name = 'lote_completo'
batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_definition_name)

### 4. Creación de Reglas (Expectations) y Suites
Definiremos el rango esperado para precios, y qué categorías son válidas.

In [12]:
exp_precio = gx.expectations.ExpectColumnValuesToBeBetween(column="price", min_value=0.01, max_value=9999)
exp_categoria = gx.expectations.ExpectColumnValuesToBeInSet(column="product_category", value_set=["Electronics", "Clothing", "Home", "Toys"])

nombre_suite = "suite_ventas_basica"
suite = gx.ExpectationSuite(name=nombre_suite)
suite = context.suites.add(suite)

suite.add_expectation(exp_precio)
suite.add_expectation(exp_categoria)

ExpectColumnValuesToBeInSet(id='2b081541-58b8-4af6-95b0-878457e08f91', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='product_category', mostly=1, row_condition=None, condition_parser=None, value_set=['Electronics', 'Clothing', 'Home', 'Toys'])

### 5. Definición de Validación (Validation Definition)

In [13]:
nombre_validacion = "validacion_diaria_ventas"
validation_definition = gx.ValidationDefinition(data=batch_definition, suite=suite, name=nombre_validacion)
validation_definition = context.validation_definitions.add(validation_definition)

### 6. Ejecución e Interpretación de Resultados
Aquí cargaremos los datos reales en Pandas y los pasaremos al flujo de validación.

In [14]:
df_real = pd.read_csv("../data/ventas_sucias.csv")
parametros_lote = {"dataframe": df_real}

resultado_validacion = validation_definition.run(batch_parameters=parametros_lote)

# Inspeccionando si la suite fue completamente exitosa
print("¿Toda la suite pasó con éxito?:", resultado_validacion.success)

Calculating Metrics:   0%|          | 0/20 [00:00<?, ?it/s]

¿Toda la suite pasó con éxito?: False


El objeto de resultado contiene amplias estadísticas para diagnosticar. Por ejemplo:

In [15]:
estadisticas = resultado_validacion.results[0].result
print(f"Total elementos evaluados: {estadisticas['element_count']}")
print(f"Valores inesperados dados: {estadisticas['unexpected_count']}")

Total elementos evaluados: 1500
Valores inesperados dados: 75


### 7. Personalización de Formatos (Customising Result Formats)

Great Expectations permite tres niveles de detalle de salida (`result_format`):
- `BASIC`: Solo muestra los conteos y porcentajes (útil para sistemas que solo deciden Pasa/No Pasa).
- `SUMMARY`: Añade una lista aleatoria de valores anómalos (por defecto, 20 ejemplos). Útil para depuración local.
- `COMPLETE`: Incluye TODOS los índices y valores anómalos. Cuidado: puede colapsar la memoria si la tabla es masiva.

Veamos cómo solicitar la versión resumida para la segunda expectativa.

In [16]:
# Modificamos la expectativa para requerir un formato SUMMARY en los resultados
suite.expectations[1].result_format = "SUMMARY"

# Guardamos el cambio en la memoria del contexto temporal
suite = context.suites.add(suite) # Sobreescribe la suite

# Corremos de nuevo
resultado_validacion_summary = validation_definition.run(batch_parameters=parametros_lote)
resultado_validacion_summary.results[1].result
print("Revisa en el JSON la sección 'partial_unexpected_list' para ver ejemplos de filas malas.")

DataContextError: Cannot add ExpectationSuite with name suite_ventas_basica because it already exists.